## Evaluating generative text models

### Using GPT to generate text

In [1]:
import torch.nn as nn
import torch
import tiktoken
from llmarch import GPTModel,generate_text_simple

In [2]:
# #[Text Generation->Text Evaluation->Training and Validation Losses]->LLM Training Function->Text gen Strategies->Weight saving and Loading->Pretrained weights from OpenAI
# Training and Validation Losses-Evaluate how well the model performs
# LLM Training Function-Train the model to generate human like text
# Text gen Strategies-Implement additional LLM text generation strategies to reduce training data memorization
# Weight saving and Loading-Implement functions to save and load the LLM weights to use or continue training the LLM later
# Pretrained weights from OpenAI-Load pretrained weights from OpenAI into our LLM Model

In [3]:
print("Hi")

Hi


In [4]:
from importlib.metadata import version
pkgs=[
    "matplotlib",
    "numpy",
    "tiktoken",
    "torch",
    "tensorflow"
]
for p in pkgs:
    print(f"{p} version:{version(p)}")

matplotlib version:3.10.8
numpy version:2.2.6
tiktoken version:0.12.0
torch version:2.7.1+cu118
tensorflow version:2.20.0


In [5]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 256, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [6]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.eval();

In [7]:
def text_to_token_ids(text,tokenizer):
    encoded=tokenizer.encode(text,allowed_special={"<|endoftext|>"})
    encoded_tensor=torch.tensor(encoded).unsqueeze(0) #unsqueeze gives it a double list(encoded) format
    return encoded_tensor

In [8]:
text="Every effort moves you"
tokenizer=tiktoken.get_encoding("gpt2")

token_ids=text_to_token_ids(text,tokenizer)
token_ids

tensor([[6109, 3626, 6100,  345]])

In [9]:
def token_ids_to_text(token_ids,tokenizer):
    flat=token_ids.squeeze(0) #reduces to single list
    return tokenizer.decode(flat.tolist())

token_ids_to_text(token_ids,tokenizer)

'Every effort moves you'

In [10]:
start_context="Hello, I am"
token_ids=generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context,tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

In [11]:
token_ids.squeeze(0).shape

torch.Size([14])

In [12]:
token_ids_to_text(token_ids,tokenizer)

'Hello, I am Laur inhab DistrinetalkQueue bear confidentlyggyenium'

### Calculate text Generation Loss(Cross entropy and Perplexity)

In [13]:
#Cross entropy 
#Now we measure the quality of each token 
inputs=torch.tensor([[16833,3626,6100],#input1-Every effort moves
                     [40,1107,588]])#input2-I really like
targets=torch.tensor([[3626,6100,345],#Target1-effort moves you
                      [1107,588,11311]])#Target2-really like chocolates


In [14]:
with torch.no_grad():
    logits=model(inputs)

In [15]:
logits.shape

torch.Size([2, 3, 50257])

In [16]:
probas=torch.softmax(logits,dim=-1)
probas.shape

torch.Size([2, 3, 50257])

In [17]:
probas #refer to slak-own

tensor([[[1.8851e-05, 1.5173e-05, 1.1687e-05,  ..., 2.2408e-05,
          6.9776e-06, 1.8775e-05],
         [9.1574e-06, 1.0062e-05, 7.8784e-06,  ..., 2.9089e-05,
          6.0105e-06, 1.3569e-05],
         [2.9875e-05, 8.8504e-06, 1.5741e-05,  ..., 3.5458e-05,
          1.4094e-05, 1.3525e-05]],

        [[1.2561e-05, 2.0537e-05, 1.4331e-05,  ..., 1.0388e-05,
          3.4784e-05, 1.4238e-05],
         [7.2733e-06, 1.7863e-05, 1.0564e-05,  ..., 2.1206e-05,
          1.1390e-05, 1.5558e-05],
         [2.9495e-05, 3.3605e-05, 4.1032e-05,  ..., 6.5251e-06,
          5.8202e-05, 1.3697e-05]]])

In [18]:
token_ids=torch.argmax(probas,dim=-1,keepdim=True)
print("Token IDs:\n",token_ids)


Token IDs:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


In [20]:
print(f"Targets 1:{token_ids_to_text(targets[0],tokenizer)}")
print(f"Predictions 1:{token_ids_to_text(token_ids[0].flatten(),tokenizer)}") #wrong output as model is untrained

Targets 1: effort moves you
Predictions 1: Armed heNetflix


In [21]:
text_idx=0
target_probas_1=probas[text_idx,[0,1,2],targets[text_idx]]

#  tensor([[[16657], 0
#          [  339], 1
#          [42826]] 2
# targets[text_idx]->highest probabilities


print("Text 1:",target_probas_1)

Text 1: tensor([7.4536e-05, 3.1061e-05, 1.1563e-05])


In [22]:
text_idx=1
target_probas_2=probas[text_idx,[0,1,2],targets[text_idx]]
print("Text 2:",target_probas_2)

Text 2: tensor([1.0337e-05, 5.6771e-05, 4.7559e-06])


In [23]:
log_probas=torch.log(torch.cat((target_probas_1,target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7765, -12.2561])


In [25]:
# token_ids_to_text(log_probas,tokenizer)

In [28]:
torch.mean(log_probas)*(-1)#---this is cross entropy loss our goal is to make it close to 0, also called negative average log probability

tensor(10.7940)

In [30]:
logits_flat=logits.flatten(0,1)
logits_flat.shape

torch.Size([6, 50257])

In [32]:
target_shape=targets.flatten()
target_shape.shape

torch.Size([6])

### Calculating Training and Validating Set Losses

In [33]:
import os
import urllib.request 
if not os.path.exists("story.txt"):
    url=("https://en.wikisource.org/wiki/The_Verdict")
    file_path="story.txt"
    urllib.request.urlretrieve(url,file_path)
with open("story.txt","r",encoding="utf-8") as f:
    raw_text=f.read()
#len(raw_text)

In [34]:
text_data=raw_text

In [35]:
text_data

'I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)\n\n"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it\'s going to send the value of my picture \'way up; but I don\'t think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing\'s lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn\'s "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?\n\nWell!--even 

In [37]:
total_chars=len(text_data)
total_chars

20479

In [38]:
total_tokens=len(tokenizer.encode(text_data))
total_tokens

5145

In [39]:
# torch.nn.functional.cross_entropy(logits)

In [40]:
tokenizer.encode(text_data)

[40,
 367,
 2885,
 1464,
 1807,
 3619,
 402,
 271,
 10899,
 2138,
 257,
 7026,
 15632,
 438,
 2016,
 257,
 922,
 5891,
 1576,
 438,
 568,
 340,
 373,
 645,
 1049,
 5975,
 284,
 502,
 284,
 3285,
 326,
 11,
 287,
 262,
 6001,
 286,
 465,
 13476,
 11,
 339,
 550,
 5710,
 465,
 12036,
 11,
 6405,
 257,
 5527,
 27075,
 11,
 290,
 4920,
 2241,
 287,
 257,
 4489,
 64,
 319,
 262,
 34686,
 41976,
 13,
 357,
 10915,
 314,
 2138,
 1807,
 340,
 561,
 423,
 587,
 10598,
 393,
 28537,
 2014,
 198,
 198,
 1,
 464,
 6001,
 286,
 465,
 13476,
 1,
 438,
 5562,
 373,
 644,
 262,
 1466,
 1444,
 340,
 13,
 314,
 460,
 3285,
 9074,
 13,
 46606,
 536,
 5469,
 438,
 14363,
 938,
 4842,
 1650,
 353,
 438,
 2934,
 489,
 3255,
 465,
 48422,
 540,
 450,
 67,
 3299,
 13,
 366,
 5189,
 1781,
 340,
 338,
 1016,
 284,
 3758,
 262,
 1988,
 286,
 616,
 4286,
 705,
 1014,
 510,
 26,
 475,
 314,
 836,
 470,
 892,
 286,
 326,
 11,
 1770,
 13,
 8759,
 2763,
 438,
 1169,
 2994,
 284,
 943,
 17034,
 318,
 477,
 314,
 892,


In [ ]:
# How to train test split:
# First 6 tokens=first training samples
# next 6 tokens- 2nd training samples

In [42]:
from textloader import create_dataloader_v1

train_ratio=0.90#90% for training
split_idx=int(train_ratio*len(text_data)) #how many total characters will go for training
train_data=text_data[:split_idx]
val_data=text_data[split_idx:]

In [43]:
torch.manual_seed(123)

train_loader=create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader=create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

In [45]:
print("Train Loader")
for x,y in train_loader:
    print(x.shape,y.shape)

Train Loader
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [46]:
print("Train Loader")
for x,y in val_loader:
    print(x.shape,y.shape)

Train Loader
torch.Size([2, 256]) torch.Size([2, 256])


In [47]:
train_tokens=0
for input_batch,target_batch in train_loader:
    train_tokens+=input_batch.numel()
train_tokens

4608

In [49]:
val_tokens=0
for input_batch,target_batch in val_loader:
    val_tokens+=input_batch.numel()
val_tokens

512

In [50]:
print("All tokens:",train_tokens+val_tokens)

All tokens: 5120


### Calculate Loss Batch and Loss Loader

In [ ]:
def calc_loss_batch(input_batch,target_batch,model,device):
    input_batch,target_batch=input_batch.to(device),target_batch.to(device) #sit on same device
    logits=model(input_batch)
    loss=torch.nn.functional.cross_entropy(logits.flatten(0,1),target_batch.flatten())
    return loss

def calc_loss_loader(data_loader,model,device,num_batches=None):
    total_loss=0
    if len(data_loader)==0:
        return float("nan")
    elif num_batches is None:
        num_batches=len(data_loader)
    else:
        num_batches=min(num_batches,len(data_loader))
    for i,(input_batch,target_batch) in enumerate(data_loader):
        if i<num_batches:
            loss=calc_loss_batch(input_batch,target_batch,model,device)
            total_loss+=loss.item() #for each batch loss gets added up
        else:
            break
    return total_loss/num_batches #iterative way to compute loss

In [52]:
torch.cuda.is_available()

True

In [55]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
model.to(device); # Whole architecture

In [58]:
len(train_loader)

9

In [ ]:
torch.manual_seed(123)
with torch.no_grad():
    train_loss=calc_loss_loader(train_loader,model,device)
    val_loss=calc_loss_loader(val_loader,model,device)  #disable gradient tracking for efficiency
train_loss,val_loss

(10.987582206726074, 10.981103897094727)

In [62]:
# Perplexity
torch.exp(torch.tensor(10.981103897094727))


tensor(58753.3750)

In [61]:
tokenizer.n_vocab

50257

In [64]:
dol=39000

In [66]:
#right now our model has high loss which pos what word it generates
print(f"Dollar to inr:{dol*int(90.77):,}")

Dollar to inr:3,510,000


# Training a LLM

In [67]:
# Iterate over train epochs->iterate over batches in each training epochs->Reset Loss Gradients from previous batch iteration -> calculate loss on current branch
                                                                                                                                                     #  |
                                                                                                                                                     #  |
# Generate sample text<-Print Training and Validation Set losses<-Update model wts using loss gradients<------|                                            / 
                                                                                                    #         |
                                                                                                    #        /                                                /
                                                                                                    #       |                                              /   
                                                                                                    #       |                                             | 
#                                                                                                           Backward pass to calculate loss gradients--<--|

In [ ]:
#RESET LOSS-RESET TO 0
#BACKWARD PASS-BACK PROP FOR LOSS GRADIENTS
#UPDATE MODEL WT PARAMETERS USING THESE LOSS GRADIENTS
#PRINT TRAINING AND VALIDATION TEST LOSSES 

In [70]:
def evaluate_model(model,train_loader,val_loader,device,eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss=calc_loss_loader(train_loader,model,device,num_batches=eval_iter)
        val_loss=calc_loss_loader(val_loader,model,device,num_batches=eval_iter)
    model.train()
    return train_loss,val_loss

In [71]:
def generate_and_print_sample(model,tokenizer,device,start_context):
    model.eval()
    context_size=model.pos_emb.weight.shape[0]
    encoded=text_to_token_ids(start_context,tokenizer).to(device)
    with torch.no_grad():
        token_ids=generate_text_simple(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )
    decoded_text=token_ids_to_text(token_ids,tokenizer)
    print(decoded_text.replace("\n"," "))
    model.train()


In [72]:
def train_model_simple(model,train_loader,val_loader,optimizer,device,num_epochs,eval_freq,eval_iter,start_context,tokenizer):
    train_losses,val_losses,track_tokens_seen=[],[],[]
    tokens_seen,global_step=0,-1

    for epoch in range(num_epochs):
        model.train()  #training mode on

        for input_batch,target_batch in train_loader:
            optimizer.zero_grad()
            loss=calc_loss_batch(input_batch,target_batch,model,device)   #Computation Graph creating 
            loss.backward()   #calculate the required loss gradients
            optimizer.step() #Update model weights using Loss Gradients[loss_grads*learning rate]
            tokens_seen+=input_batch.numel()
            global_step+=1

            if global_step%eval_freq==0:
                train_loss,val_loss=evaluate_model(
                    model,train_loader,val_loader,device,eval_iter
                )
                train_losses.append(train_loss)
                val_losses.append(train_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train Loss {train_loss:.3f},Val loss {val_loss:.3f}")
        generate_and_print_sample(   #Sample text after each topic
            model,tokenizer,device,start_context
        )
    return train_losses,val_losses,track_tokens_seen

In [73]:
torch.manual_seed(123)
model=GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=0.0004,weight_decay=0.1) #Adam Optimizer-with weight decay

In [75]:
num_ep=25
train_losses,val_losses,tokens_seen=train_model_simple(
    model,train_loader,val_loader,optimizer,device,num_epochs=num_ep,eval_freq=5,eval_iter=5,start_context="Every effort moves you",tokenizer=tokenizer
)

Ep 1 (Step 000000): Train Loss 0.455,Val loss 6.410
Ep 1 (Step 000005): Train Loss 0.356,Val loss 6.463
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back the window-curtains, I may be pardoned the bull--that I found
Ep 2 (Step 000010): Train Loss 0.262,Val loss 6.545
Ep 2 (Step 000015): Train Loss 0.200,Val loss 6.620
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I
Ep 3 (Step 000020): Train Loss 0.166,Val loss 6.695
Ep 3 (Step 000025): Train Loss 0.114,Val loss 6.756
Every effort moves you?"  "Yes--quite insensible to the irony. She wanted him vindicated--and by me!"  He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I
Ep 4 (Step 000030): Train Loss 0.122,Val loss 6.818
Ep 4 (Step 000035): 